In [6]:
from Exoplanet import Exoplanet
from astroquery.mast import Catalogs

data_paths = []

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    data_paths.extend([file_name.strip() for file_name in file.readlines()])
    file.seek(0)

for path in data_paths:
    e = Exoplanet.load_from_json(path)

    data = Catalogs.query_object(f'TIC {e.id_num}', catalog = 'TIC')[0]

    e.add_host_star_attributes(
        radius = data['rad'],
        mass = data['mass'],
        teff = data['Teff']
    )

    e.calculate_attributes()

    print(f'{e.id_num}: {e.mass}')

    e.to_json(f'Exoplanet Discovering/Paper Materials/Data/{e.host_star['name']} Exoplanet {e.letter}.json')

with open('Exoplanet Discovering/candidate_data_paths.txt', 'r+') as file:
    file.truncate(0)

11799872: 365.75064362858825
23689332: 370.64024653353033
35655828: 3018.373800858723
35655828: 106.90334413520871
38850860: 453.97373286412795
55655482: 396.0133083149033
55727842: 14.467633955867637
62573638: 579.9618257307278
73248090: 1307.2460161695483
76903841: 6193.419156733383
90083037: 592.7137336305241
102625324: 376.83069851192124
176871438: 732.7619775598712
234518605: 667.3795231862669
268290940: 42.68902337190331
270462117: 490.6800006365983
279322914: 610.9189583655561
284611896: nan
295440174: 976.7216223370542
297701617: 3192.307577655116
300291839: 59.47665946039712
306635088: 970.9670830412982
342556321: 887.3742728222841
375477302: 433.38232297150984
441444256: nan
451596415: 66.98167658118143
467113954: 1019.2159541666332
734505586: 41.01342546577284


In [12]:
import os
import json

data = []

for (dirpath, dirnames, filenames) in os.walk('Exoplanet Discovering/Paper Materials/Data'):
    for filepath in filenames:
        if filepath == '.DS_Store' or filepath.endswith('.csv'):
            continue

        with open(dirpath + '/' + filepath, 'r') as file:
            exoplanet = json.loads(file.read())

        if type(exoplanet) == list:
            continue

        data.append(exoplanet)

    break

data = sorted(data, key = lambda x: x['name'])

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'w') as file:
    string = json.dumps(data, indent = 4)
    string = string.replace('NaN', 'null')
    string = string.replace('nan', '')

    file.write(string)

In [13]:
import json

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

with open('Exoplanet Discovering/Paper Materials/params_planet_20250503_001.txt', 'w') as file:
    string = ''
    for exoplanet in data:
        string += exoplanet['csv_string'] + '\n'

    file.write(string)

In [1]:
import json
import pandas
import math

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

data = pandas.DataFrame([exoplanet['parameters'] for exoplanet in data])
data['target'] = data['target'].apply(lambda x: x.replace('TIC', 'TIC '))
data = data.set_index('target')
data.replace('', math.nan, inplace = True)
data = data.drop(['tag', 'r_planet', 'flag'], axis = 1)
data['disp'] = data['disp'].mask(data['radius'] >= 23.53885947 + 1, 'FP') # z-score radius limit of all candidates/planets, from Nasa Exoplanet Archive
data = data.sort_values(['disp', 'target'
                         ], ascending = [False, True])

data = data.drop(['TIC ' + str(ticid) + '.01' for ticid in [
        284611896,
        268290940,
        62573638,
        23689332,
        441444256,
        234518605,
        306635088,
        451596415]])

columns = ['disp', 'period', 'period_unc', 'epoch', 'epoch_unc', 'radius', 'mass', 'temp', 'depth', 'depth_unc', 'duration', 'duration_unc', 'sma']
data = data.loc[:, columns]

columns = data.select_dtypes(include = 'number').columns.to_list()
for value in ['period', 'depth', 'duration', 'epoch']:
    columns.remove(value)

data[columns] = data[columns].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (3 - 1) - int(math.floor(math.log10(abs(x)))))
)

data[['period', 'depth', 'duration']] = data[['period', 'depth', 'duration']].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (6 - 1) - int(math.floor(math.log10(abs(x)))))
)

data['epoch'] = data['epoch'].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (9 - 1) - int(math.floor(math.log10(abs(x)))))
)

print(data['mass'].astype(int).to_csv(index = False))
print(data.loc[data['disp'] == 'PC', ['disp', 'mass', 'radius']])

data.replace(math.nan, 'null', inplace = True)
data.to_csv('Exoplanet Discovering/Paper Materials/Data/ctois.csv')

mass
377
366
491
59
107
433
454
396
14
41
593
733
611
977
3190
887
3020
1020
1310
6190

                 disp   mass  radius
target                              
TIC 102625324.01   PC  377.0   17.50
TIC 11799872.01    PC  366.0   14.70
TIC 270462117.01   PC  491.0   22.00
TIC 300291839.01   PC   59.5    7.63
TIC 35655828.02    PC  107.0    8.76
TIC 375477302.01   PC  433.0   20.20
TIC 38850860.01    PC  454.0   20.90
TIC 55655482.01    PC  396.0   13.10
TIC 55727842.01    PC   14.5    3.31
TIC 734505586.01   PC   41.0    6.82
TIC 90083037.01    PC  593.0   24.40


In [10]:
import pandas
import json

with open('Exoplanet Discovering/Paper Materials/Data/full.json', 'r') as file:
    data = json.loads(file.read())

ticids = [exoplanet['id_num'] for exoplanet in data]

data = pandas.DataFrame()
for ticid in ticids:
    url = f'https://exofop.ipac.caltech.edu/tess/download_planet.php?id={ticid}'

    data = pandas.concat([data, pandas.read_csv(url, delimiter = '|')])

data = data.set_index('Name')
data = data.sort_index()
data

,Table,ID,TOI,Disposition,Epoch (BJD),Epoch (BJD) Error,Period (days),Period (days) Error,Depth (mmag),Depth (mmag) Error,...,Time of Periastron (BJD),Time of Periastron (BJD) Error,Velocity Semi-amplitude (m/s),Velocity Semi-amplitude (m/s) Error,SNR,Date,User,Group,Tag,Notes
Name,,,,,,,,,,,,,,,,,,,,,
TIC 102625324.01,CTOI,NaN,NaN,PC,2.458546e+06,0.021953,9.176416,0.177373,13.460065,22.377276,...,NaN,NaN,NaN,NaN,NaN,2025-04-01 17:19:25,kzhu,NaN,449168,"Planetary Candidate, Kevin Zhu"
TIC 11799872.01,CTOI,NaN,NaN,PC,2.458843e+06,0.047250,4.258437,0.108448,6.920025,0.155288,...,NaN,NaN,NaN,NaN,NaN,2025-04-13 10:36:13,kzhu,NaN,452929,From TCE reviewed by Kevin Zhu
TIC 176871438.01,CTOI,NaN,NaN,PC,2.457134e+07,0.042594,15.138331,0.033465,10.152198,0.375116,...,NaN,NaN,NaN,NaN,NaN,2025-03-30 13:10:31,kzhu,NaN,449068,Possible exoplanet; u-shaped; some transits no...
TIC 234518605.01,CTOI,NaN,NaN,PC,2.458327e+06,0.006975,5.679047,5.689360,64.324769,2.885382,...,NaN,NaN,NaN,NaN,NaN,2025-04-09 15:35:10,kzhu,NaN,452904,"Planetary Candidate, Kevin Zhu"
TIC 23689332.01,CTOI,NaN,NaN,PC,2.459346e+06,0.040013,17.297518,12.682299,9.642561,0.454132,...,NaN,NaN,NaN,NaN,NaN,2025-04-02 20:36:08,kzhu,NaN,451929,"Planetary Candidate, Kevin Zhu"
TIC 268290940.01,CTOI,NaN,NaN,PC,2.459966e+06,0.007088,9.306725,0.060706,47.858383,0.772199,...,NaN,NaN,NaN,NaN,NaN,2025-03-30 12:45:03,kzhu,NaN,449066,Possible exoplanet; v-shaped; no secondary ecl...
TIC 270462117.01,CTOI,NaN,NaN,PC,2.460017e+06,0.019891,3.932376,0.038741,9.842996,0.826952,...,NaN,NaN,NaN,NaN,NaN,2025-04-06 09:36:55,kzhu,NaN,452563,"Planetary Candidate, Kevin Zhu"
TIC 279322914.01,CTOI,NaN,NaN,PC,2.458338e+06,0.019249,18.868943,0.030847,33.940846,1.359135,...,NaN,NaN,NaN,NaN,NaN,2025-04-01 16:47:36,kzhu,NaN,449165,"Planetary Candidate, Kevin Zhu"
TIC 284611896.01,CTOI,NaN,NaN,PC,2.458770e+06,0.026303,8.179997,0.034166,40.631075,0.794857,...,NaN,NaN,NaN,NaN,NaN,2025-03-29 13:07:53,kzhu,NaN,449009,wobble/noise in lc; estimate star with F-star ...
